### TF-IDF 중요한 이유
- 문서에 자주 언급되는 단어가 중요한 단어는 아니다
- A문서에서는 자주 언급되지만 B문서에는 드문 단어가 A문서에 핵심 단어
- 예) 기자라는 단어는 어는 뉴스 본문에나 등장하지만 -> 중요단어는 아니다
- 문서의 핵심 단어 추출에 스임 -> 검색(BM25)하이브리드 RAG 구축

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from kiwipiepy import Kiwi

plt.rcParams["font.family"] = "Malgun Gothic"   # 한글 깨짐 방지(윈도우 기본 폰트)
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("../data/11-1_뉴스정제.csv")
kiwi = Kiwi()

In [4]:
# 명사 추출 함수 만들기

def extract_nonus(text):
    nonus =[]
    result = kiwi.tokenize(text)
    for token in result:
        if token.tag.startswith("N"): # N으로 시작하는 글자가 명사
            nonus.append(token.form) # 실제명사만 넣기

    filtered_nouns = []
    for noun in nonus : 
         if len(noun)> 1:
                filtered_nouns.append(noun)
    return filtered_nouns

### 1. TF-IDF 개념
- TF(단어 빈도) : 한 문서에는 그 단어가 자주 나올수록 높다
- IDF(역문서 빈도) : 그 단어가 여러 문서에 흔할수록 낮아진다.(흔한 단어에 페널티, 벌점)
- TF-IDF = TF x IDF : 이 문서에는 자주 나오지만.. 다른 문서에는 드물수록 점수가 높다


In [29]:
docs = ["자동차 가격 자동차", #문서1
        "자동차 가격 가격", #문서2
        "날씨 날씨 자동차"] #문서3

vocab = ["자동차",'가격','날씨']
data = pd.DataFrame({
    "자동차" : [2, 1,1],
    "가격" : [1, 2,0],
    "날씨" : [0, 0,2],
})
data.head()

,자동차,가격,날씨
0,2,1,0
1,1,2,0
2,1,0,2


In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from kiwipiepy import Kiwi

plt.rcParams["font.family"] = "Malgun Gothic"   # 한글 깨짐 방지(윈도우 기본 폰트)
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("../data/11-1_뉴스정제.csv")
kiwi = Kiwi()

In [30]:
#전체 문서 수
import numpy as np

N = 3
df_ = (data > 0).sum() # 데이터 영역은 아님 - 각 단어가 등장한 문서 수
idf = np.log( N / df_)
idf

자동차    0.000000
가격     0.405465
날씨     1.098612
dtype: float64

In [27]:
# 사이킷런 패키지를 이용해서 tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer


In [32]:
vec = TfidfVectorizer(tokenizer=extract_nonus, token_pattern=None)
X = vec.fit_transform(df['정제본문'])
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 100830 stored elements and shape (1000, 14444)>

In [23]:
X.data

array([0.70710678, 0.70710678])

In [35]:
feature_words = vec.get_feature_names_out()
print(feature_words)

['1이더리움' '2여객터미널' '3사' ... '힘겨루기' '힙밥' '힙합']


In [33]:
print("문서 수 X 단어수", X.shape)


문서 수 X 단어수 (1000, 14444)


In [36]:
len(feature_words)

14444

In [37]:
#한 기사에서 가장 tf-idf 값이 높은 단어를 찾는 함수 생성
def keywords(doc_idx, count=5): #문서의 넘버를 받아서 가장 높은 단어를 몇개 볼지정해주는 함수
    row = X[doc_idx].toarray()[0]
    top = row.argsort()[-count:][::-1]
    return [feature_words[i] for i in top]

In [44]:
keywords(0)

['현대백화점그룹', '광주광역시', '서울', '여의도', '현대']

In [42]:
keywords(1)

['이스타항공', '회생', '의원', '관계', '오해']

In [43]:
keywords(2)

['주화', '이벤트', '주년', '농협은행', '인스타그램']

In [41]:
keywords(3)

['재정부', '기획', '완화', '대출', '유류']